# Question 8

**Why** nightly batch in  **Warehouse**  is a kind of pain ?
**ans** -  It takes to much time  ETL time , and if there is failure in and step then  need to hard rollback to remove the corrupt data which again takes the too much time.

**How** **Lakehouse** prevent this problem ? 
**ans**  - In **Lakehouse** we can use the Delta Lake to prevent this problem using  micro batch streaming  by following the medellion architecture (Bronze , here all the raw data -> silver, strict schema and clean data -> gold (business logic)


# Operational Risk Reduction: ACID Transactions & Time Travel

**Operational Risk in Traditional Batch **Warehouse****

ACID Transactions - Partial Writes & Reader Contention: If a nightly batch fails halfway through, tables contain half-baked data. Analysts reading tables during loads encounter locks or dirty reads. 

Time Travel  - Destructive Overwrites & Hard Rollbacks: Recovering from an incorrect batch write requires restoring from full database backups, which takes hours/days and halts production.

**How **Lakehouse** Mitigation Works**

ACID Transactions - Atomicity & Isolation: Delta/Iceberg uses optimistic concurrency control with log-based commits (JSON/AVRO logs). Readers see a consistent snapshot; writes are atomic—if a batch fails, no partial data is visible.

Time Travel - Versioned Commits & Easy Rollbacks: Every operation creates a new version in the transaction log. Rolling back an bad load is as simple as querying or restoring VERSION AS OF n-1.




# Question 9

How the Transaction Log Resolves Order & Handles Conflicts
Resolution of Write Order:

Delta Lake uses Optimistic Concurrency Control (OCC).

When Writer 1 and Writer 2 start, they both read version 0 of the table.

Writer 1 finishes first and writes log file 00000000000000000001.json.

Writer 2 attempts to write 00000000000000000001.json. It sees that version 1 already exists, checks if its append conflicts with version 1, and automatically retries by updating its read state to version 1 and committing as 00000000000000000002.json.

What Happens During Conflicts?

Appends (No Conflict): Since both jobs are performing APPEND operations without reading the data to perform mutations (like UPDATE or DELETE), Delta resolves the conflict automatically via optimistic retry.

Write-Write Conflicts (Concurrent Updates/Deletes): If Writer 1 deletes/updates a file while Writer 2 tries to update or read that same file, Delta throws a ConcurrentAppendException or ConcurrentTransactionException. One job succeeds; the other fails and must be retried.


### Concurrent Write Test

* Created a Databricks Job with **two parallel tasks**.
* Both tasks were configured to **insert data into the same Delta table**.
* The **first task completed successfully**.
* The **second task failed with a concurrency error (`DELTA_METADATA_CHANGED`)** because the table metadata was changed by the concurrent transaction.
* This demonstrates how **Delta Lake handles concurrent writes using its transaction log and concurrency control**.



# Concurrent Task Test

Two different notebooks were created to test concurrent writes on the same Delta table.

### Concurrent Task 1
[Open Concurrent Task 1 Notebook]('https://dbc-2bd5e4bd-405a.cloud.databricks.com/editor/notebooks/3233653864340257?o=7474659560736194')

### Concurrent Task 2
[Open Concurrent Task 2 Notebook]('https://dbc-2bd5e4bd-405a.cloud.databricks.com/editor/notebooks/3233653864340258?o=7474659560736194')

# Quesiton 10

# 3 Concrete Advantages of a **Lakehouse** for Analysts

 Near-Real-Time Access to Raw and Curated Data.

 
****Warehouse****: Analysts wait for overnight ETL pipelines to transform and load data before querying.

****Lakehouse****: Streaming/micro-batch ingestion makes Bronze (raw) and Silver (curated) tables available within minutes without waiting for end-of-day batch runs.

Querying Unstructured, Semi-Structured, and Structured Data Together

**Warehouse**: Proprietary formats usually limit analysts to structured tables or rigid JSON columns, requiring heavy pre-processing.

**Lakehouse**: Analysts can query Parquet/Delta tables, JSON logs, geospatial data, images, or audio metadata directly using standard SQL or Python/R notebooks in a single platform.

Time Travel and Auditability for Reproducible Analytics

**Warehouse**: Auditing historical state requires snapshot tables or complex Slowly Changing Dimension (SCD) modeling.

**Lakehouse**: Analysts can query past snapshots (VERSION AS OF or TIMESTAMP AS OF) to reproduce historical reports, verify data discrepancies, or debug pipeline changes instantly.

1 Tradeoff to Watch For
Increased Complexity in Small-File Management and Optimization

Continuous streams and frequent appends generate thousands of small Parquet files in object storage, which degrades SQL query performance if left unmanaged.

Mitigation: Requires automated maintenance routines (e.g., running OPTIMIZE and VACUUM in Delta Lake) to compact small files and index data using Z-Ordering/Liquid Clustering.